# MD Simulation Workflow for HSA Binding Pose Identification

This notebook performs molecular dynamics (MD) simulation of a protein–ligand complex using
[OpenMM](https://openmm.org/) with AMBER force fields (ff19SB + GAFF2 + TIP3P), following the
workflow structure of [making-it-rain](https://github.com/pablo-arantes/making-it-rain)
adapted for local PC execution with a CUDA GPU.

## Workflow overview

1. **Ligand preparation** — SDF → PDB (OpenBabel), formal charge check (RDKit)
2. **GAFF2 parameterization** — antechamber, parmchk2, tleap (AMBERTools)
3. **System setup** — protein–ligand complex solvation with TIP3P (tleap)
4. **Equilibration MD** — 0.5 ns NPT, 310 K, 1 bar, position restraints
5. **Production MD** — 5 × 1 ns NPT = 5 ns total, 310 K, 1 bar
6. **Post-processing** — trajectory concatenation, conversion to XTC
7. **Trajectory analysis** — Cα RMSD, Cα RMSF, radius of gyration, ligand–protein distance
8. **Interaction energy** — nonbonded (electrostatic + vdW) ligand–protein interaction energy
9. **MM/PBSA & MM/GBSA** — binding free energy estimation (AMBERTools MMPBSA.py)

## Prerequisites

- AMBERTools ≥ 23 (`antechamber`, `parmchk2`, `tleap`, `cpptraj`, `pdb4amber`)
- OpenMM ≥ 8, OpenFF Toolkit, ParmEd, MDAnalysis, ProLIF, PyTraj
- OpenBabel (`obabel`)
- Python packages: see `environment.yml`
- CUDA-capable GPU (change platform to `CPU` or `OpenCL` if unavailable)

## Third-party attribution

The equilibration/production simulation cells are adapted from **making-it-rain**
([pablo-arantes/making-it-rain](https://github.com/pablo-arantes/making-it-rain),
MIT License). See `THIRD_PARTY_NOTICES.md` in the repository root.

## Input files required

- `<PDB_ID>-processed.pdb` — receptor PDB (prepared, hydrogens added, waters removed)
- `<POSE_TAG>.sdf` — docked ligand pose SDF from GNINA (e.g., `pose01.sdf`)

## 1. User-defined parameters

Set all parameters in this cell before running the notebook.

In [ ]:
import os
import sys
import math
import fnmatch
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

warnings.filterwarnings("ignore", category=UserWarning)

# ── Paths ─────────────────────────────────────────────────────────────────
# Working directory: all output files will be written here.
# Run from the folder containing the input files, or set an absolute path.
WORK_DIR = Path(".")                         # change if needed

# ── Identifiers ───────────────────────────────────────────────────────────
PDB_ID   = "1E7A"    # PDB ID of the receptor (change to your PDB ID)
POSE_TAG = "pose01"  # docking pose identifier (change to your pose)

# Processed receptor PDB (protonated at pH 7.4, waters removed)
RECEPTOR_PDB = WORK_DIR / f"{PDB_ID}-processed.pdb"

# Docked ligand SDF (one pose; from GNINA pose*.sdf)
LIGAND_SDF   = WORK_DIR / f"{POSE_TAG}.sdf"

# ── Force-field parameters ────────────────────────────────────────────────
FORCE_FIELD   = "ff19SB"  # protein force field: "ff19SB" or "ff14SB"
WATER_TYPE    = "TIP3P"   # water model: "TIP3P" or "OPC"
BOX_SIZE_A    = 12        # solvation box padding (Angstroms)
SALT          = "NaCl"    # salt type: "NaCl" or "KCl"
CONCENTRATION = 0.15      # salt concentration (Molar)

# ── Equilibration parameters ──────────────────────────────────────────────
EQUIL_TIME_NS     = 0.5   # equilibration simulation time (ns)
EQUIL_DT_FS       = 2     # integration timestep (fs)
EQUIL_TEMP_K      = 310   # temperature (K)
EQUIL_PRESS_BAR   = 1     # pressure (bar)
EQUIL_FC_KJMOL    = 700   # position restraint force constant (kJ/mol)
EQUIL_TRAJ_PS     = 10    # trajectory write frequency (ps)
EQUIL_LOG_PS      = 10    # log write frequency (ps)
MINIM_STEPS       = 20000 # energy minimization steps

# ── Production MD parameters ──────────────────────────────────────────────
PROD_STRIDE_NS  = 1       # length of each MD stride (ns)
PROD_N_STRIDES  = 5       # number of strides → total = PROD_STRIDE_NS * PROD_N_STRIDES ns
PROD_DT_FS      = 2       # integration timestep (fs)
PROD_TEMP_K     = 310     # temperature (K)
PROD_PRESS_BAR  = 1       # pressure (bar)
PROD_TRAJ_PS    = 10      # trajectory write frequency (ps)
PROD_LOG_PS     = 10      # log write frequency (ps)

# ── GPU / platform ────────────────────────────────────────────────────────
# Options: "CUDA" (recommended), "OpenCL", "CPU"
PLATFORM_NAME = "CUDA"
PRECISION     = "mixed"   # "single", "mixed", or "double"

# ── Residue number offset ──────────────────────────────────────────────────
# Auto-detect residue offset from the processed PDB.
# Some PDB files start from residue 2 or 3 (missing N-terminal residues);
# this offset corrects internal numbering to match PDB standard numbering.
def _get_pdb_offset(pdb_path):
    """Read the first ATOM record and return (first_resseq - 1)."""
    try:
        with open(pdb_path) as f:
            for line in f:
                if line.startswith("ATOM"):
                    return int(line[22:26].strip()) - 1
    except Exception as e:
        print(f"WARNING: Could not read PDB for offset: {e}")
    return 0

PDB_OFFSET = _get_pdb_offset(RECEPTOR_PDB)
print(f"PDB offset : {PDB_OFFSET} (first residue in PDB = {PDB_OFFSET + 1})")

# ── Derived convenience paths ─────────────────────────────────────────────
WORK_DIR.mkdir(parents=True, exist_ok=True)
workDir = str(WORK_DIR)

print(f"PDB_ID      : {PDB_ID}")
print(f"POSE_TAG    : {POSE_TAG}")
print(f"Receptor    : {RECEPTOR_PDB}")
print(f"Ligand SDF  : {LIGAND_SDF}")
print(f"Work dir    : {WORK_DIR.resolve()}")
print(f"Total prod  : {PROD_STRIDE_NS * PROD_N_STRIDES} ns ({PROD_N_STRIDES} × {PROD_STRIDE_NS} ns)")

## 2. Ligand preparation

Converts the docked ligand SDF to PDB format (OpenBabel) and checks the formal charge
(RDKit). The formal charge is needed for GAFF2 parameterization in the next step.

> **Note on hydrogen addition**: If OpenBabel-generated hydrogens are geometrically
> incorrect, use [hydride](https://github.com/biotite-dev/hydride) in a separate
> conda environment:  
> `conda activate hydride`  
> `hydride --charges 7.4 -i ligand.sdf -o ligand_hydride.sdf`  
> Then set `LIGAND_SDF` above to the hydride-corrected file.


In [ ]:
from openbabel import openbabel
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem.Draw import IPythonConsole
import builtins
from IPython.display import display

LIGAND_PDB  = WORK_DIR / "ligand_input.pdb"
LIGAND_H_PDB = WORK_DIR / "ligand_H2.pdb"

# SDF → PDB via OpenBabel (hydrogens already in SDF)
obConv = openbabel.OBConversion()
obConv.SetInAndOutFormats("sdf", "pdb")
mol_ob = openbabel.OBMol()
obConv.ReadFile(mol_ob, str(LIGAND_SDF))
obConv.WriteFile(mol_ob, str(LIGAND_PDB))

# Read with RDKit to check formal charge and display 2D structure
mol = Chem.MolFromPDBFile(str(LIGAND_PDB), removeHs=False)
Chem.MolToPDBFile(mol, str(LIGAND_H_PDB))

formal_q = builtins.sum(a.GetFormalCharge() for a in mol.GetAtoms())
CHARGE = formal_q
print(f"Formal charge of ligand: {CHARGE}")
print(f"(Set CHARGE manually below if the RDKit value is incorrect.)")

# Display 2D structure
mol_2d = Chem.MolFromPDBFile(str(LIGAND_H_PDB), removeHs=False)
for atom in mol_2d.GetAtoms():
    atom.SetAtomMapNum(atom.GetIdx())
display(mol_2d)


In [ ]:
# If RDKit formal charge is wrong, override here:
# CHARGE = -1   # e.g., for deprotonated carboxylate
print(f"Using ligand net charge: {CHARGE}")


## 3. GAFF2 parameterization (AMBERTools)

Generates GAFF2 force-field parameters for the ligand using:
- `pdb4amber` — PDB atom renaming
- `antechamber` — AM1-BCC charge assignment and mol2 generation
- `parmchk2` — missing parameter generation (frcmod)
- `tleap` — saves `.lib` and renumbered GAFF2 PDB

Outputs: `ligand.mol2`, `ligand.frcmod`, `lig.lib`, `ligand_gaff.pdb`


In [ ]:
LIGAND_H2_PDB   = WORK_DIR / "ligand_H2.pdb"
LIGAND_MOL2     = WORK_DIR / "ligand.mol2"
LIGAND_FRCMOD   = WORK_DIR / "ligand.frcmod"
LIGAND_LIB      = WORK_DIR / "lig.lib"
LIGAND_GAFF_PDB = WORK_DIR / "ligand_gaff.pdb"
PROTEIN_LIGAND  = WORK_DIR / "protein_ligand.pdb"
STARTING1_PDB   = WORK_DIR / "starting1.pdb"
STARTING_END_PDB = WORK_DIR / "starting_end.pdb"

ff = "leaprc.protein.ff19SB" if FORCE_FIELD == "ff19SB" else "leaprc.protein.ff14SB"
water = "leaprc.water.tip3p" if WATER_TYPE == "TIP3P" else "leaprc.water.opc"
water_box = "TIP3PBOX" if WATER_TYPE == "TIP3P" else "OPCBOX"

# Step 1: pdb4amber
cmd1 = f"pdb4amber -i {LIGAND_H2_PDB} -o {WORK_DIR / 'ligand_h.pdb'}"
# Step 2: antechamber (AM1-BCC charges)
cmd2 = (f"antechamber -i {WORK_DIR / 'ligand_h.pdb'} -fi pdb "
        f"-o {LIGAND_MOL2} -fo mol2 -c bcc -nc {CHARGE} -rn LIG -at gaff2")
# Step 3: parmchk2
cmd3 = f"parmchk2 -i {LIGAND_MOL2} -f mol2 -o {LIGAND_FRCMOD} -s gaff2"

# Step 4: tleap to generate lib and GAFF2 PDB
TLEAP_IN = WORK_DIR / "tleap_gaff2.in"
TLEAP_IN.write_text(
    f"source {ff}\n"
    f"source leaprc.gaff2\n"
    f"LIG = loadmol2 {LIGAND_MOL2}\n"
    f"loadamberparams {LIGAND_FRCMOD}\n"
    f"saveoff LIG {LIGAND_LIB}\n"
    f"savepdb LIG {LIGAND_GAFF_PDB}\n"
    f"quit\n"
)

for cmd in [cmd1, cmd2, cmd3]:
    print(f"Running: {cmd}")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"  STDERR: {res.stderr.strip()[:500]}")
    else:
        print(f"  OK")

res4 = subprocess.run(f"tleap -f {TLEAP_IN}", shell=True, capture_output=True, text=True)
print("tleap (GAFF2 lib):", "OK" if res4.returncode == 0 else f"FAILED: {res4.stderr[:300]}")


## 4. System setup (protein–ligand complex, solvation)

Prepares the protein structure with `cpptraj/prepareforleap` and `pdb4amber`,
combines it with the GAFF2-parameterized ligand, and solvates with TIP3P water
at 0.15 M NaCl using `tleap`.

Outputs: `SYS_gaff2.prmtop`, `SYS_gaff2.crd`, `SYS.pdb`


In [ ]:
from biopandas.pdb import PandasPdb

# Step 1: cpptraj prepareforleap — removes waters, adds missing H, renumbers
PREPAREFORLEAP_IN = WORK_DIR / "prepareforleap.in"
PREPAREFORLEAP_IN.write_text(
    f"parm {RECEPTOR_PDB}\n"
    f"loadcrd {RECEPTOR_PDB} name edited\n"
    f"prepareforleap crdset edited name from-prepareforleap \\\n"
    f"pdbout {STARTING1_PDB} nowat noh\n"
    f"go\n"
)
res = subprocess.run(f"cpptraj -i {PREPAREFORLEAP_IN}", shell=True, capture_output=True, text=True)
print("cpptraj prepareforleap:", "OK" if res.returncode == 0 else f"FAILED\n{res.stderr[:500]}")

# Step 2: pdb4amber — renumber and fix atom names
cmd_p4a = f"pdb4amber -i {STARTING1_PDB} -o {STARTING_END_PDB} -a"
res = subprocess.run(cmd_p4a, shell=True, capture_output=True, text=True)
print("pdb4amber:", "OK" if res.returncode == 0 else f"FAILED\n{res.stderr[:300]}")

# Step 3: Combine protein + ligand PDB
with open(PROTEIN_LIGAND, "w") as out_f:
    for src in [STARTING_END_PDB, LIGAND_GAFF_PDB]:
        with open(src) as in_f:
            for line in in_f:
                if not line.startswith("END"):
                    out_f.write(line)
    out_f.write("END\n")

# Clean PDB (remove non-ATOM/HETATM records using biopandas)
ppdb = PandasPdb().read_pdb(str(PROTEIN_LIGAND))
ppdb.to_pdb(path=str(PROTEIN_LIGAND), records=["ATOM", "HETATM"], gz=False, append_newline=True)
print(f"Combined protein+ligand PDB: {PROTEIN_LIGAND}")


In [ ]:
# tleap: solvate + ions + save topology
SYS_TOP  = WORK_DIR / "SYS_gaff2.prmtop"
SYS_CRD  = WORK_DIR / "SYS_gaff2.crd"
SYS_PDB  = WORK_DIR / "SYS.pdb"
SYS_NW_TOP = WORK_DIR / "SYS_nw.prmtop"
SYS_NW_CRD = WORK_DIR / "SYS_nw.crd"
SYS_NW_PDB = WORK_DIR / "SYS_nw.pdb"
TLEAP_FULL = WORK_DIR / "tleap_full.in"

# Determine ion numbers for target concentration (0.15 M NaCl)
pos_ion = "Na+" if SALT == "NaCl" else "K+"

# Preliminary tleap run to get box volume
prelim_tleap = WORK_DIR / "tleap_prelim.in"
prelim_tleap.write_text(
    f"source {ff}\nsource leaprc.DNA.OL15\nsource leaprc.RNA.OL3\n"
    f"source leaprc.GLYCAM_06j-1\nsource leaprc.gaff2\nsource {water}\n"
    f"loadamberparams {LIGAND_FRCMOD}\nloadoff {LIGAND_LIB}\n"
    f"SYS = loadpdb {PROTEIN_LIGAND}\nalignaxes SYS\n"
    f"savepdb SYS {SYS_NW_PDB}\nsaveamberparm SYS {SYS_NW_TOP} {SYS_NW_CRD}\n"
    f"solvatebox SYS {water_box} {BOX_SIZE_A} 0.7\n"
    f"saveamberparm SYS {SYS_TOP} {SYS_CRD}\nsavepdb SYS {SYS_PDB}\nquit\n"
)
res = subprocess.run(f"tleap -f {prelim_tleap}", shell=True, capture_output=True, text=True)

# Parse box volume from leap.log
leap_log = (WORK_DIR / "leap.log").read_text() if (WORK_DIR / "leap.log").exists() else res.stdout
vol = None
for line in leap_log.splitlines():
    if "Volume:" in line:
        try:
            vol = float(line.split()[1])
        except (IndexError, ValueError):
            pass
if vol is None:
    print("WARNING: Could not parse box volume from leap.log; using default ion count 101.")
    num_ion = 101
else:
    vol_lit  = vol * 1e-27
    atom_lit = 9.03e22
    num_ion  = max(0, int(vol_lit * (CONCENTRATION / 0.15) * atom_lit))
print(f"Box volume: {vol} Å³ → {num_ion} {pos_ion} + {num_ion} Cl- added")

# Final tleap with correct ion count
TLEAP_FULL.write_text(
    f"source {ff}\nsource leaprc.DNA.OL15\nsource leaprc.RNA.OL3\n"
    f"source leaprc.GLYCAM_06j-1\nsource leaprc.gaff2\nsource {water}\n"
    f"loadamberparams {LIGAND_FRCMOD}\nloadoff {LIGAND_LIB}\n"
    f"SYS = loadpdb {PROTEIN_LIGAND}\nalignaxes SYS\n"
    f"check SYS\ncharge SYS\n"
    f"addions SYS {pos_ion} 0\naddions SYS Cl- 0\n"
    f"check SYS\ncharge SYS\n"
    f"savepdb SYS {SYS_NW_PDB}\nsaveamberparm SYS {SYS_NW_TOP} {SYS_NW_CRD}\n"
    f"solvatebox SYS {water_box} {BOX_SIZE_A} 0.7\n"
    f"addIonsRand SYS {pos_ion} {num_ion} Cl- {num_ion}\n"
    f"saveamberparm SYS {SYS_TOP} {SYS_CRD}\nsavepdb SYS {SYS_PDB}\nquit\n"
)
res = subprocess.run(f"tleap -f {TLEAP_FULL}", shell=True, capture_output=True, text=True)

if SYS_PDB.exists() and SYS_TOP.exists() and SYS_CRD.exists():
    print("System topology generated successfully.")
else:
    print("ERROR: tleap failed. Check leap.log for details.")
    print(res.stderr[:500])


## 5. Equilibration MD (NPT, 0.5 ns)

Runs energy minimization followed by NPT equilibration with position restraints
on all protein + ligand heavy atoms (force constant = 700 kJ/mol).

**Force field**: ff19SB (protein) + GAFF2 (ligand) + TIP3P (water)  
**Integrator**: Langevin (310 K, friction 1 ps⁻¹)  
**Barostat**: Monte Carlo (1 bar)  
**Non-bonded**: PME, 10 Å cutoff, HBonds constraint

> **GPU note**: Change `PLATFORM_NAME` to `"CPU"` or `"OpenCL"` if CUDA is unavailable.


In [ ]:
import openmm as mm
from openmm import *
from openmm.app import *
from openmm.unit import *
import pytraj as pt

# ── Helper functions ───────────────────────────────────────────────────────
def restraints(system, crd, fc, restraint_array):
    """Add positional restraints to heavy atoms."""
    if fc > 0:
        posres = CustomExternalForce('k*periodicdistance(x, y, z, x0, y0, z0)^2;')
        posres.addPerParticleParameter('k')
        posres.addPerParticleParameter('x0')
        posres.addPerParticleParameter('y0')
        posres.addPerParticleParameter('z0')
        for atom_idx in restraint_array:
            atom_idx = int(atom_idx)
            pos = crd.positions[atom_idx].value_in_unit(nanometers)
            posres.addParticle(atom_idx, [fc, pos[0], pos[1], pos[2]])
        system.addForce(posres)
    return system

# ── Equilibration parameters ───────────────────────────────────────────────
equil_jobname    = str(WORK_DIR / "prot_lig_equil")
time_ps          = float(EQUIL_TIME_NS) * 1000
simulation_time  = float(time_ps) * picosecond
dt               = int(EQUIL_DT_FS) * femtosecond
temperature      = float(EQUIL_TEMP_K) * kelvin
pressure         = float(EQUIL_PRESS_BAR) * bar
savcrd_freq      = int(EQUIL_TRAJ_PS) * picosecond
print_freq       = int(EQUIL_LOG_PS) * picosecond
restraint_fc     = int(EQUIL_FC_KJMOL)

nsteps  = int(simulation_time.value_in_unit(picosecond) / dt.value_in_unit(picosecond))
nprint  = int(print_freq.value_in_unit(picosecond) / dt.value_in_unit(picosecond))
nsavcrd = int(savcrd_freq.value_in_unit(picosecond) / dt.value_in_unit(picosecond))
friction = 1.0

print(f"Equilibration: {EQUIL_TIME_NS} ns, {nsteps} steps, T={EQUIL_TEMP_K} K, P={EQUIL_PRESS_BAR} bar")

# ── Build system ───────────────────────────────────────────────────────────
prmtop = AmberPrmtopFile(str(SYS_TOP))
inpcrd = AmberInpcrdFile(str(SYS_CRD))

system = prmtop.createSystem(
    nonbondedMethod=PME,
    nonbondedCutoff=1.0 * nanometers,
    constraints=HBonds,
    rigidWater=True,
    ewaldErrorTolerance=0.0005,
)
system.addForce(MonteCarloBarostat(pressure, temperature))

# Position restraints (heavy atoms only, exclude WAT/Na+/Cl-)
pt_system = pt.iterload(str(SYS_CRD), str(SYS_TOP))
restraint_array = pt.select_atoms('!(:H*) & !(:WAT) & !(:Na+) & !(:Cl-) & !(:Mg+) & !(:K+)', pt_system.top)
system = restraints(system, inpcrd, restraint_fc, restraint_array)

# ── Integrator and platform ────────────────────────────────────────────────
integrator = LangevinIntegrator(temperature, friction, dt)
integrator.setConstraintTolerance(1e-6)

try:
    platform = mm.Platform.getPlatformByName(PLATFORM_NAME)
    properties = {"Precision": PRECISION} if PLATFORM_NAME in ("CUDA", "OpenCL") else {}
    simulation = Simulation(prmtop.topology, system, integrator, platform, properties)
    print(f"Using platform: {PLATFORM_NAME} ({PRECISION} precision)")
except Exception as e:
    print(f"Platform '{PLATFORM_NAME}' unavailable ({e}); falling back to CPU.")
    simulation = Simulation(prmtop.topology, system, integrator)

simulation.context.setPositions(inpcrd.positions)
if inpcrd.boxVectors is not None:
    simulation.context.setPeriodicBoxVectors(*inpcrd.boxVectors)

# ── Energy minimization ────────────────────────────────────────────────────
print(f"Energy minimization: {MINIM_STEPS} steps ...")
simulation.minimizeEnergy(tolerance=10 * kilojoule/mole/nanometer, maxIterations=MINIM_STEPS)
E = simulation.context.getState(getEnergy=True).getPotentialEnergy()
print(f"Potential energy after minimization: {E}")
simulation.context.setVelocitiesToTemperature(temperature)

# ── Run equilibration ──────────────────────────────────────────────────────
dcd_file = equil_jobname + ".dcd"
log_file = equil_jobname + ".log"
rst_file = equil_jobname + ".rst"
pdb_file = equil_jobname + ".pdb"

dcd_rep = DCDReporter(dcd_file, nsavcrd)
firstdcdstep = nsteps + nsavcrd
dcd_rep._dcd = DCDFile(dcd_rep._out, simulation.topology, simulation.integrator.getStepSize(), firstdcdstep, nsavcrd)
simulation.reporters.append(dcd_rep)
simulation.reporters.append(StateDataReporter(sys.stdout, nprint, step=True, speed=True, progress=True,
                                              totalSteps=nsteps, remainingTime=True, separator='\t'))
simulation.reporters.append(StateDataReporter(log_file, nprint, step=True, kineticEnergy=True,
                                              potentialEnergy=True, totalEnergy=True,
                                              temperature=True, volume=True, speed=True))
print(f"\nRunning equilibration ({nsteps} steps) ...")
simulation.step(nsteps)
simulation.reporters.clear()

# Save state and last frame
state = simulation.context.getState(getPositions=True, getVelocities=True)
with open(rst_file, 'w') as f:
    f.write(XmlSerializer.serialize(state))
positions = simulation.context.getState(getPositions=True).getPositions()
PDBFile.writeFile(simulation.topology, positions, open(pdb_file, 'w'))
print(f"\nEquilibration complete.  State: {rst_file}  Last frame: {pdb_file}")


## 6. Production MD (NPT, 5 × 1 ns)

Runs 5 independent 1 ns strides for a total of 5 ns production simulation.
Each stride saves a checkpoint (`.rst`) so the run can be resumed if interrupted.
No position restraints are applied.


In [ ]:
prod_jobname  = str(WORK_DIR / "prot_lig_prod")
equil_rst     = WORK_DIR / "prot_lig_equil.rst"
equil_pdb     = WORK_DIR / "prot_lig_equil.pdb"

stride_time_ps = float(PROD_STRIDE_NS) * 1000
stride_time    = float(stride_time_ps) * picosecond
nstride        = int(PROD_N_STRIDES)
dt             = int(PROD_DT_FS) * femtosecond
temperature    = float(PROD_TEMP_K) * kelvin
pressure       = float(PROD_PRESS_BAR) * bar
savcrd_freq    = int(PROD_TRAJ_PS) * picosecond
print_freq     = int(PROD_LOG_PS) * picosecond
friction       = 1.0

nsteps  = int(stride_time.value_in_unit(picosecond) / dt.value_in_unit(picosecond))
nprint  = int(print_freq.value_in_unit(picosecond) / dt.value_in_unit(picosecond))
nsavcrd = int(savcrd_freq.value_in_unit(picosecond) / dt.value_in_unit(picosecond))

print(f"Production: {PROD_N_STRIDES} strides × {PROD_STRIDE_NS} ns = {PROD_N_STRIDES * PROD_STRIDE_NS} ns total")
print(f"Steps per stride: {nsteps}  |  DT: {PROD_DT_FS} fs  |  T: {PROD_TEMP_K} K  |  P: {PROD_PRESS_BAR} bar")

# Build system (no position restraints)
prmtop = AmberPrmtopFile(str(SYS_TOP))
inpcrd = AmberInpcrdFile(str(SYS_CRD))
system = prmtop.createSystem(
    nonbondedMethod=PME,
    nonbondedCutoff=1.0 * nanometers,
    constraints=HBonds,
    rigidWater=True,
    ewaldErrorTolerance=0.0005,
)
system.addForce(MonteCarloBarostat(pressure, temperature))
integrator = LangevinIntegrator(temperature, friction, dt)
integrator.setConstraintTolerance(1e-6)

try:
    platform = mm.Platform.getPlatformByName(PLATFORM_NAME)
    properties = {"Precision": PRECISION} if PLATFORM_NAME in ("CUDA", "OpenCL") else {}
    simulation = Simulation(prmtop.topology, system, integrator, platform, properties)
except Exception as e:
    print(f"Platform '{PLATFORM_NAME}' unavailable ({e}); falling back to CPU.")
    simulation = Simulation(prmtop.topology, system, integrator)

simulation.context.setPositions(inpcrd.positions)
if inpcrd.boxVectors is not None:
    simulation.context.setPeriodicBoxVectors(*inpcrd.boxVectors)

# ── Stride loop ────────────────────────────────────────────────────────────
for n in range(1, nstride + 1):
    rst_file = f"{prod_jobname}_{n}.rst"
    prv_rst  = f"{prod_jobname}_{n-1}.rst" if n > 1 else str(equil_rst)
    dcd_file = f"{prod_jobname}_{n}.dcd"
    log_file = f"{prod_jobname}_{n}.log"
    pdb_file = f"{prod_jobname}_{n}.pdb"

    if Path(rst_file).exists():
        print(f"Stride {n}: already done (rst exists). Skipping.")
        continue

    print(f"\n>>> Stride {n}/{nstride} — loading state from {Path(prv_rst).name} <<<")
    with open(prv_rst) as f:
        simulation.context.setState(XmlSerializer.deserialize(f.read()))
    currstep = (n - 1) * nsteps
    simulation.currentStep = currstep
    simulation.context.setTime(currstep * dt.in_units_of(picosecond))

    dcd_rep = DCDReporter(dcd_file, nsavcrd)
    dcd_rep._dcd = DCDFile(dcd_rep._out, simulation.topology, simulation.integrator.getStepSize(),
                           currstep + nsavcrd, nsavcrd)
    simulation.reporters.append(dcd_rep)
    simulation.reporters.append(StateDataReporter(sys.stdout, nprint, step=True, speed=True, progress=True,
                                                  totalSteps=nsteps * nstride, remainingTime=True, separator='\t'))
    simulation.reporters.append(StateDataReporter(log_file, nprint, step=True, kineticEnergy=True,
                                                  potentialEnergy=True, totalEnergy=True,
                                                  temperature=True, volume=True, speed=True))
    print(f"Running {nsteps} steps ...")
    simulation.step(nsteps)
    simulation.reporters.clear()

    state = simulation.context.getState(getPositions=True, getVelocities=True)
    with open(rst_file, 'w') as f:
        f.write(XmlSerializer.serialize(state))
    positions = simulation.context.getState(getPositions=True).getPositions()
    PDBFile.writeFile(simulation.topology, positions, open(pdb_file, 'w'))
    print(f"Stride {n} done.  State saved: {rst_file}")

print("\n>>> All production strides complete. <<<")


## 7. Post-processing: trajectory concatenation and conversion

Concatenates production DCD strides, removes water and ions, auto-images the trajectory,
and converts to XTC format (for external analysis tools).


In [ ]:
import pytraj as pt
import MDAnalysis as mda

# Remove water topology
gaff_top = pt.load_topology(str(SYS_TOP))
gaff_nw = gaff_top['!:WAT']
nw_top_file = str(WORK_DIR / "SYS_gaff2_nw.prmtop")
gaff_nw.save(nw_top_file)
print(f"No-water topology: {nw_top_file}")

# Concatenate strides
template = str(WORK_DIR / "prot_lig_prod_%s.dcd")
flist = [template % str(i) for i in range(1, PROD_N_STRIDES + 1)]
trajlist = pt.load(flist, str(SYS_TOP), stride=1)

# Strip water, auto-image, align to frame 0
nw_xtc = str(WORK_DIR / f"prot_lig_prod1-{PROD_N_STRIDES}_nw.xtc")
t0 = trajlist.strip(':WAT')
traj_image = t0.iterframe(autoimage=True, rmsfit=0)
pt.write_traj(nw_xtc, traj_image, overwrite=True, options="xtc")

traj_load = pt.load(nw_xtc, nw_top_file)
print(traj_load)
print(f"Concatenated trajectory: {nw_xtc}")

# Simulation analysis variables
simulation_ns = float(PROD_STRIDE_NS) * PROD_N_STRIDES
Write_the_trajectory = str(PROD_TRAJ_PS)
stride_traj = 1


## 8. Trajectory analysis: RMSD, RMSF, radius of gyration

Computes standard MD quality metrics from the concatenated trajectory.


In [ ]:
time_ps  = traj_load.n_frames * int(Write_the_trajectory)
time_arr = np.arange(0, time_ps / 1000, int(Write_the_trajectory) / 1000)

# ── CA RMSD ───────────────────────────────────────────────────────────────
rmsd = pt.rmsd(traj_load, ref=0, mask="@CA")
plt.figure()
plt.plot(time_arr[:len(rmsd)], rmsd, alpha=0.7, color='blue', linewidth=0.8)
plt.xlabel("Time (ns)"); plt.ylabel("RMSD (Å)")
plt.title(f"{PDB_ID} {POSE_TAG} — Cα RMSD")
plt.tight_layout()
plt.savefig(str(WORK_DIR / "rmsd_ca.png"), dpi=150); plt.show()

rmsd_df = pd.DataFrame({"time_ns": time_arr[:len(rmsd)], "rmsd_ca_A": rmsd})
rmsd_df.to_csv(str(WORK_DIR / "rmsd_ca.csv"), index=False)
print("Mean Cα RMSD:", np.mean(rmsd).round(3), "Å")

In [ ]:
# ── CA RMSF ───────────────────────────────────────────────────────────────
rmsf = pt.rmsf(traj_load, "@CA")
plt.figure()
plt.plot(rmsf[:, 0].astype(int) + PDB_OFFSET, rmsf[:, 1], alpha=0.9, color='red', linewidth=0.8)
plt.xlabel("Residue number (PDB)"); plt.ylabel("RMSF (Å)")
plt.title(f"{PDB_ID} {POSE_TAG} — Cα RMSF")
plt.tight_layout()
plt.savefig(str(WORK_DIR / "rmsf_ca.png"), dpi=150); plt.show()

rmsf_df = pd.DataFrame({
    "residue_index": rmsf[:, 0].astype(int),
    "residue_pdb": (rmsf[:, 0].astype(int) + PDB_OFFSET),  # PDB standard numbering
    "rmsf_ca_A": rmsf[:, 1],
})
rmsf_df.to_csv(str(WORK_DIR / "rmsf_ca.csv"), index=False)

In [ ]:
# ── Radius of gyration ────────────────────────────────────────────────────
radgyr = pt.radgyr(traj_load, mask="@CA")
plt.figure()
plt.plot(time_arr[:len(radgyr)], radgyr, alpha=0.7, color='green', linewidth=0.8)
plt.xlabel("Time (ns)"); plt.ylabel("Radius of gyration (Å)")
plt.title(f"{PDB_ID} {POSE_TAG} — Radius of gyration")
plt.tight_layout()
plt.savefig(str(WORK_DIR / "radius_gyration.png"), dpi=150); plt.show()

radgyr_df = pd.DataFrame({"time_ns": time_arr[:len(radgyr)], "radgyr_ca_A": radgyr})
radgyr_df.to_csv(str(WORK_DIR / "radius_gyration.csv"), index=False)

### 8.4 Distance between the ligand and surrounding residues

Calculates the average distance between the ligand and the protein residues that are within 5 Å of the ligand in the initial trajectory frame.

In [ ]:
# ── Distance to nearest residues ──────────────────────────────────────────────
cutoff_dist = 5.0  # Å

# Define reference as the first frame
pt_topology = traj_load.top
pt_topology.set_reference(traj_load[0])

# Select residues within cutoff from the ligand (excluding water/ions/ligand itself)
indices = pt_topology.select(f'(:LIG<:{cutoff_dist})&!(:WAT|:Na+,Cl-,LIG)')
residues = [res.original_resid for res in pt_topology[indices].residues]
res_string = ','.join(str(e) for e in residues)
print(f"Selected residues (within {cutoff_dist} Å): {res_string}")

if res_string:
    mask = f":LIG :{res_string}"
    dist = pt.distance(traj_load, mask)
    dist_mean = np.mean(dist)
    dist_std = np.std(dist)
    print(f"Average distance: {dist_mean:.2f} ± {dist_std:.2f} Å")

    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(time_arr[:len(dist)], dist, color='springgreen', lw=1.0)
    plt.xlim(0, PROD_NS_TOTAL)
    plt.xlabel("Time (ns)")
    plt.ylabel(r"Distance ($\AA$)")
    plt.title("Distance between ligand and contact residues")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.savefig(str(WORK_DIR / "distance.png"), dpi=300, bbox_inches='tight')
    plt.show()

    # Save CSV
    dist_df = pd.DataFrame({
        "time_ns": time_arr[:len(dist)],
        "distance_A": dist
    })
    dist_df.to_csv(str(WORK_DIR / "distance.csv"), index=False)
else:
    print("No residues found within the cutoff distance.")


## 9. Nonbonded interaction energy (ligand–protein)

Computes nonbonded interaction energy between the ligand (`:LIG`) and the rest of the system
using PyTraj. This is **not** a binding free energy; it is used as a stability indicator.


In [ ]:
pt_topology = traj_load.top
restraint_array_ie = pt.select_atoms('!(:WAT) & !(:Na+) & !(:Cl-)', pt_topology)

eelec = pt.energy_decomposition(traj_load, igb=0, dtype='dict').get('eelec', None)
evdw  = pt.energy_decomposition(traj_load, igb=0, dtype='dict').get('evdw',  None)

# If energy_decomposition is unavailable, use pytraj calc_energy or nativecontacts as fallback
# Here we save placeholder if needed:
if eelec is not None and evdw is not None:
    ie_df = pd.DataFrame({"time_ns": time_arr[:len(eelec)], "eelec": eelec, "evdw": evdw})
    ie_df["total_ie"] = ie_df["eelec"] + ie_df["evdw"]
    ie_df.to_csv(str(WORK_DIR / "Interaction_energy.csv"), index=False)
    # Individual eelec/evdw CSVs (required by 02_add_md_features)
    pd.DataFrame({"time_ns": time_arr[:len(eelec)], "eelec": eelec}).to_csv(
        str(WORK_DIR / "Interaction_energy_eelec.csv"), index=False)
    pd.DataFrame({"time_ns": time_arr[:len(evdw)], "evdw": evdw}).to_csv(
        str(WORK_DIR / "Interaction_energy_evdw.csv"), index=False)

    plt.figure()
    plt.plot(time_arr[:len(eelec)], ie_df["total_ie"], alpha=0.7, linewidth=0.8)
    plt.xlabel("Time (ns)"); plt.ylabel("Interaction energy (kcal/mol)")
    plt.title(f"{PDB_ID} {POSE_TAG} — Ligand–protein interaction energy")
    plt.tight_layout()
    plt.savefig(str(WORK_DIR / "Interaction_energy.png"), dpi=150); plt.show()
    print("Interaction energy CSV saved.")
else:
    print("energy_decomposition not available; skipping interaction energy calculation.")

## 10. MM/PBSA and MM/GBSA binding free energy

Estimates binding free energy using **`MMPBSA.py`** from AMBERTools.
Both MM/GBSA (GB model, igb=2) and MM/PBSA (PB model) are calculated in a single run.

The output file `FINAL_RESULTS_MMPBSA.dat` is required by `02_add_md_features.ipynb`
to extract `md_mmgbsa_delta_total` and `md_mmpbsa_delta_total` features.

> **Note**: This step requires `ante-MMPBSA.py` and `MMPBSA.py` from AMBERTools.


In [ ]:
# ── MM/PBSA and MM/GBSA calculation ──────────────────────────────────────
# Parameters
IGB           = 2       # GB model: 2 (OBC) recommended
SALT_CONC_MM  = 0.15    # salt concentration for MM/PBSA (M)
MMPBSA_OUT    = "FINAL_RESULTS_MMPBSA"  # output filename prefix

# Radii selection based on igb
RADII_MAP = {1: "mbondi", 2: "mbondi2", 5: "mbondi2", 7: "bondi", 8: "mbondi3"}
mbondi = RADII_MAP.get(IGB, "mbondi2")

# Use the concatenated no-water trajectory
traj_for_mmpbsa = str(WORK_DIR / f"prot_lig_prod1-{PROD_N_STRIDES}_nw.xtc")
prmtop_full     = str(WORK_DIR / "SYS_gaff2.prmtop")
final_mmpbsa    = str(WORK_DIR / MMPBSA_OUT)

# Estimate number of frames and set stride
import pytraj as pt
_temp_traj = pt.iterload(traj_for_mmpbsa, str(WORK_DIR / "SYS_gaff2_nw.prmtop"))
n_frames_mm = _temp_traj.n_frames
stride_mm   = max(1, n_frames_mm // 10)
print(f"Trajectory frames: {n_frames_mm}, MMPBSA stride: {stride_mm}")

# Write mmpbsa.in
mmpbsa_in = str(WORK_DIR / "mmpbsa.in")
with open(mmpbsa_in, "w") as f:
    f.write(f"""&general
  endframe={n_frames_mm}, interval={stride_mm}, strip_mask=:WAT:Na+:Cl-:Mg+:K+,
/
&gb
  igb={IGB}, saltcon={SALT_CONC_MM},
/
&pb
  istrng={SALT_CONC_MM}, inp=2, radiopt=0, prbrad=1.4,
/
""")
print(f"Wrote {mmpbsa_in}")

# Run ante-MMPBSA.py and MMPBSA.py
ante_cmd = (
    f"ante-MMPBSA.py -p {prmtop_full} -c com.prmtop -r rec.prmtop "
    f"-l ligand.prmtop -s :WAT:Na+:Cl-:Mg+:K+ -n :LIG --radii {mbondi}"
)
mmpbsa_cmd = (
    f"MMPBSA.py -O -i {mmpbsa_in} -o {final_mmpbsa}.dat "
    f"-sp {prmtop_full} -cp com.prmtop -rp rec.prmtop -lp ligand.prmtop "
    f"-y {traj_for_mmpbsa}"
)

print("Running ante-MMPBSA.py ...")
r1 = subprocess.run(ante_cmd, shell=True, capture_output=True, text=True,
                    cwd=str(WORK_DIR))
if r1.returncode != 0:
    print(f"ante-MMPBSA STDERR:\n{r1.stderr}")
    raise RuntimeError("ante-MMPBSA.py failed")
print("  done.")

print("Running MMPBSA.py (this may take several minutes) ...")
r2 = subprocess.run(mmpbsa_cmd, shell=True, capture_output=True, text=True,
                    cwd=str(WORK_DIR))
if r2.returncode != 0:
    print(f"MMPBSA STDERR:\n{r2.stderr}")
    raise RuntimeError("MMPBSA.py failed")
print("  done.")

# Move intermediate files to a subdirectory
mmpbsa_dir = WORK_DIR / f"MMPBSA_igb_{IGB}"
mmpbsa_dir.mkdir(exist_ok=True)
import glob as _glob
for pat in ["_MMPBSA*", "com.prmtop", "rec.prmtop", "ligand.prmtop", "reference.frc"]:
    for fp in _glob.glob(str(WORK_DIR / pat)):
        Path(fp).rename(mmpbsa_dir / Path(fp).name)

# Display results
result_file = f"{final_mmpbsa}.dat"
if os.path.isfile(result_file):
    with open(result_file) as f:
        print(f.read())
else:
    print(f"WARNING: {result_file} not found")


## 11. Summary of output files

| File | Description |
|---|---|
| `SYS_gaff2.prmtop` / `.crd` | AMBER topology + coordinates |
| `SYS.pdb` | Solvated system PDB |
| `prot_lig_equil.dcd/.pdb/.rst` | Equilibration trajectory / last frame / state |
| `prot_lig_prod_N.dcd/.pdb/.rst` | Production stride N trajectory / last frame / state |
| `prot_lig_prod1-5_nw.xtc` | Concatenated no-water trajectory (XTC) |
| `SYS_gaff2_nw.prmtop` | No-water topology |
| `rmsd_ca.csv` | Cα RMSD time series |
| `rmsf_ca.csv` | Cα RMSF per residue (with PDB residue numbering) |
| `radius_gyration.csv` | Radius of gyration time series |
| `distance.csv` | Ligand–protein contact distance time series |
| `Interaction_energy.csv` | Ligand–protein total interaction energy |
| `Interaction_energy_eelec.csv` | Electrostatic interaction energy |
| `Interaction_energy_evdw.csv` | Van der Waals interaction energy |
| `FINAL_RESULTS_MMPBSA.dat` | MM/GBSA + MM/PBSA binding free energy |

These outputs feed into the feature extraction notebook (`02_add_md_features.ipynb`) in the analysis pipeline.
